<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_11/tugas_praktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tugas Praktikum**
1. Buatlah model SVM dengan menggunakan data voice.csv dengan ketentuan<br>
  a. Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.<br>
    - Gunakan model dengan kernel linier.
    - Gunakan model dengan kernel polynomial.
    - Gunakan model dengan kernel RBF.<br>
  b. Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.




In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [2]:
# 1. Load dataset
df = pd.read_csv('voice.csv')

# Pisahkan fitur (X) dan target (y)
X = df.drop('label', axis=1)
y = df['label']

In [3]:
# Standardisasi fitur untuk performa SVM optimal
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Pengujian dengan rasio 70:30 dan 80:20 serta 3 kernel
ratios = [0.3, 0.2]  # test_size
kernels = ['linear', 'poly', 'rbf']

results = []

for test_size in ratios:
    split_name = '70:30' if test_size == 0.3 else '80:20'

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )

    for kernel in kernels:
        # Buat dan latih model SVM
        model = SVC(kernel=kernel)
        model.fit(X_train, y_train)

        # Prediksi dan hitung akurasi
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio Split': split_name,
            'Kernel': kernel.capitalize(),
            'Akurasi': f"{acc:.4f}"
        })

# 3. Tabulasikan hasil performansi
df_results = pd.DataFrame(results)
print(df_results)

  Rasio Split  Kernel Akurasi
0       70:30  Linear  0.9706
1       70:30    Poly  0.9569
2       70:30     Rbf  0.9811
3       80:20  Linear  0.9763
4       80:20    Poly  0.9685
5       80:20     Rbf  0.9826



- Pengaruh Rasio Split Data: Pembagian data dengan rasio 80:20 secara konsisten menghasilkan tingkat akurasi yang lebih tinggi dibanding rasio 70:30 pada seluruh jenis kernel. Hal ini disebabkan oleh bertambahnya porsi data latih sehingga model dapat mempelajari pola data secara lebih optimal.

- Pengaruh Jenis Kernel: Kernel RBF (Radial Basis Function) mencatatkan performansi tertinggi pada kedua rasio split dengan akurasi maksimal 98.26% pada rasio 80:20. Kernel Linear berada pada posisi kedua, sedangkan kernel Polynomial memiliki tingkat akurasi terendah di antara ketiganya.

2. Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

In [9]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.image as mpimg
import zipfile
import os
from google.colab import files
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Upload & Ekstrak Dataset Otomatis
if not os.path.exists("images/training"):
    if not os.path.exists("images.zip"):
        print("Silakan pilih dan upload file 'images.zip':")
        uploaded = files.upload()

    if os.path.exists("images.zip"):
        with zipfile.ZipFile("images.zip", 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Dataset berhasil diekstrak!")

# Load Dataset
def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []
    for dir_path in dirs:
        if dir_path.is_dir():
            label = dir_path.name
            for file in dir_path.glob('*.jpg'):
                img = mpimg.imread(file)
                if img is not None:
                    img_list.append((img, label))
    return img_list

train_dir = "images/training/"
train_img = load_dataset(train_dir)
print(f"Jumlah gambar berhasil dimuat: {len(train_img)}")

# Ekstraksi Fitur Histogram Citra
def extract_histogram(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [8], [0, 180])
    hist_s = cv2.calcHist([hsv], [1], None, [8], [0, 256])
    hist_v = cv2.calcHist([hsv], [2], None, [8], [0, 256])

    hist = np.concatenate([hist_h, hist_s, hist_v]).flatten()
    total = np.sum(hist)
    if total > 0:
        hist = hist / total
    return hist

# Ekstraksi Fitur & Labeling
X = []
y = []

for img, label in train_img:
    features = extract_histogram(img)
    X.append(features)
    y.append(label)

X = np.array(X)
y = np.array(y)

# Split Data (Rasio 80:20) & Standarisasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model SVM RBF Baseline
model_rbf = SVC(kernel='rbf', random_state=42)
model_rbf.fit(X_train_scaled, y_train)

y_pred = model_rbf.predict(X_test_scaled)
acc_baseline = accuracy_score(y_test, y_pred)
print(f"Akurasi Baseline RBF: {acc_baseline * 100:.2f}%")

# Hyperparameter Tuning (GridSearchCV)
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

# Evaluasi Model Terbaik
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)
acc_best = accuracy_score(y_test, y_pred_best)

print("\n--- Hasil Tuning ---")
print(f"Parameter Terbaik : {grid_search.best_params_}")
print(f"Akurasi Setelah Tuning: {acc_best * 100:.2f}%")

Silakan pilih dan upload file 'images.zip':


Saving images.zip to images.zip
Dataset berhasil diekstrak!
Jumlah gambar berhasil dimuat: 240
Akurasi Baseline RBF: 100.00%

--- Hasil Tuning ---
Parameter Terbaik : {'C': 1, 'gamma': 'scale'}
Akurasi Setelah Tuning: 100.00%


Ekstraksi fitur histogram pada ruang warna HSV terbukti sangat efektif untuk membedakan kondisi siang dan malam. Pengujian menunjukkan bahwa model baseline RBF dengan parameter standar (𝐶 = 1 C=1, 𝛾 = ’scale’ γ=’scale’) sudah mampu mencapai akurasi sempurna (100.00%) pada data uji. Hasil Grid Search mengonfirmasi bahwa konfigurasi parameter ini merupakan pilihan optimal untuk dataset ini.